# Geometry-Only Wrong Spherical Mapping

This notebook isolates the screen-geometry problem from the phase-sensitive simple-cell problem.

The question here is purely geometric:

- Assume **no phase sensitivity** and no tuning-estimation artifact.
- Quantify how much **wrong spherical mapping** changes local orientation geometry during session-to-session gaze shifts.
- Compare:
  - the intended exact spherical correction,
  - exact spherical correction with the wrong eye-to-screen distance,
  - no spherical correction on the screen plane.

We focus on gaze shifts from $0^\circ$ to $10^\circ$ and vary the effective field-of-view center across the screen.

## Outline

1. Import numerical and plotting libraries.
2. Define camera, retinal, and session parameters.
3. Implement the reference spherical mapping.
4. Implement wrong spherical mapping models.
5. Parameterize effective FOV centers.
6. Generate gaze shifts from $0^\circ$ to $10^\circ$.
7. Project image coordinates under both mappings.
8. Compute angular, spatial, radial, and tangential mapping errors.
9. Simulate multi-session drift across starting FOV centers.
10. Aggregate errors by shift magnitude and session.
11. Visualize error surfaces and session-wise drift.
12. Compare sensitivity across FOV center offsets.
13. Fit simple error-growth models.
14. Export summary tables and reproducible artifacts.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils import load_config, ensure_output_dirs, project_root
from src.plotting import set_plot_style

set_plot_style()
np.random.seed(7)

ROOT = project_root()
CONFIG = load_config("configs/default.yaml", debug=True)
PATHS = ensure_output_dirs(CONFIG)
NOTEBOOK_NAME = "geometry_only_wrong_spherical_mapping"
TABLE_DIR = PATHS["tables"]
FIG_DIR = PATHS["figures"]

print(ROOT)
print(TABLE_DIR)
print(FIG_DIR)

/Users/tobiasr/Documents/GitHub/model-gabordrift
/Users/tobiasr/Documents/GitHub/model-gabordrift/results/tables
/Users/tobiasr/Documents/GitHub/model-gabordrift/results/figures


## 2. Define Camera, Retinal, and Session Parameters

We work in physical screen coordinates because the distance error lives in the camera/screen geometry, not in the neural response model.

- `true_distance_cm` is the intended eye-to-screen-center distance.
- `assumed_distances_cm` are the corrected and mis-corrected distances.
- `gaze_magnitudes_deg` spans the session drift range from $0^\circ$ to $10^\circ$.
- `nominal_patch_extent_deg` defines the local image region around each effective FOV center used for spatial displacement summaries.

No phase-sensitive variables are introduced anywhere in this notebook.

In [2]:
true_distance_cm = 15.0
assumed_distances_cm = {
    "exact_true_distance": true_distance_cm,
    "exact_distance_minus_5cm": true_distance_cm - 5.0,
    "exact_distance_plus_5cm": true_distance_cm + 5.0,
}

screen_half_extent_deg = 50.0
nominal_patch_extent_deg = 8.0
patch_resolution = 41
jacobian_eps_cm = 0.02
orientation_grid_deg = np.arange(0.0, 180.0, 5.0)
gaze_magnitudes_deg = np.linspace(0.0, 10.0, 21)
session_count = len(gaze_magnitudes_deg)

gaze_paths = {
    "horizontal": np.column_stack([gaze_magnitudes_deg, np.zeros_like(gaze_magnitudes_deg)]),
    "vertical": np.column_stack([np.zeros_like(gaze_magnitudes_deg), gaze_magnitudes_deg]),
    "diagonal": np.column_stack([gaze_magnitudes_deg / np.sqrt(2.0), gaze_magnitudes_deg / np.sqrt(2.0)]),
}

parameter_table = pd.DataFrame(
    {
        "parameter": [
            "true_distance_cm",
            "screen_half_extent_deg",
            "nominal_patch_extent_deg",
            "patch_resolution",
            "session_count",
            "gaze_shift_max_deg",
        ],
        "value": [
            true_distance_cm,
            screen_half_extent_deg,
            nominal_patch_extent_deg,
            patch_resolution,
            session_count,
            float(gaze_magnitudes_deg.max()),
        ],
    }
)
parameter_table

,parameter,value
0,true_distance_cm,15.0
1,screen_half_extent_deg,50.0
2,nominal_patch_extent_deg,8.0
3,patch_resolution,41.0
4,session_count,21.0
5,gaze_shift_max_deg,10.0


## 3–4. Implement Reference and Wrong Spherical Mapping Models

The reference model uses the exact gnomonic screen-to-angle and angle-to-screen transforms.

The wrong models are:

- exact spherical correction with the wrong distance,
- planar no-correction on the screen plane.

The exact formulas are:

$$
\mathrm{az}(x) = \arctan\left(\frac{x}{d}\right)
$$

$$
\mathrm{el}(x, y) = \arctan\left(\frac{y}{\sqrt{d^2 + x^2}}\right)
$$

and the inverse gnomonic projection is

$$
x = d\tan(\mathrm{az}), \qquad y = d\,\frac{\tan(\mathrm{el})}{\cos(\mathrm{az})}.
$$

A key prediction is that a wrong distance inside the exact inverse should mainly introduce isotropic scale error, while skipping the spherical correction should introduce local orientation distortion away from the screen center.

In [3]:
def screen_cm_to_angles_deg(x_cm, y_cm, distance_cm):
    x = np.asarray(x_cm, dtype=float)
    y = np.asarray(y_cm, dtype=float)
    d = float(distance_cm)
    az = np.rad2deg(np.arctan2(x, d))
    el = np.rad2deg(np.arctan2(y, np.sqrt(d**2 + x**2)))
    return az, el


def angles_deg_to_screen_cm_exact(az_deg, el_deg, distance_cm):
    az = np.deg2rad(np.asarray(az_deg, dtype=float))
    el = np.deg2rad(np.asarray(el_deg, dtype=float))
    d = float(distance_cm)
    x = d * np.tan(az)
    y = d * np.tan(el) / np.maximum(np.cos(az), 1e-12)
    return x, y


def angles_deg_to_screen_cm_planar(az_deg, el_deg, distance_cm):
    az = np.deg2rad(np.asarray(az_deg, dtype=float))
    el = np.deg2rad(np.asarray(el_deg, dtype=float))
    d = float(distance_cm)
    return d * az, d * el


def mapping_from_true_screen(x_cm, y_cm, *, true_distance_cm, scenario):
    az_deg, el_deg = screen_cm_to_angles_deg(x_cm, y_cm, true_distance_cm)
    if scenario == "exact_true_distance":
        return angles_deg_to_screen_cm_exact(az_deg, el_deg, true_distance_cm)
    if scenario == "exact_distance_minus_5cm":
        return angles_deg_to_screen_cm_exact(az_deg, el_deg, true_distance_cm - 5.0)
    if scenario == "exact_distance_plus_5cm":
        return angles_deg_to_screen_cm_exact(az_deg, el_deg, true_distance_cm + 5.0)
    if scenario == "planar_no_correction":
        return angles_deg_to_screen_cm_planar(az_deg, el_deg, true_distance_cm)
    raise ValueError(f"Unknown scenario: {scenario}")


def orientation_difference_deg(a_deg, b_deg):
    return ((np.asarray(a_deg) - np.asarray(b_deg) + 90.0) % 180.0) - 90.0


def numerical_jacobian(center_x_cm, center_y_cm, *, true_distance_cm, scenario, eps_cm):
    x0 = float(center_x_cm)
    y0 = float(center_y_cm)
    eps = float(eps_cm)
    fx1, fy1 = mapping_from_true_screen(x0 + eps, y0, true_distance_cm=true_distance_cm, scenario=scenario)
    fx0, fy0 = mapping_from_true_screen(x0 - eps, y0, true_distance_cm=true_distance_cm, scenario=scenario)
    gx1, gy1 = mapping_from_true_screen(x0, y0 + eps, true_distance_cm=true_distance_cm, scenario=scenario)
    gx0, gy0 = mapping_from_true_screen(x0, y0 - eps, true_distance_cm=true_distance_cm, scenario=scenario)
    return np.array(
        [
            [(fx1 - fx0) / (2.0 * eps), (gx1 - gx0) / (2.0 * eps)],
            [(fy1 - fy0) / (2.0 * eps), (gy1 - gy0) / (2.0 * eps)],
        ],
        dtype=float,
    )


def transformed_orientation_deg(jacobian, orientation_deg):
    theta = np.deg2rad(float(orientation_deg))
    vec = np.array([np.cos(theta), np.sin(theta)], dtype=float)
    mapped = np.asarray(jacobian, dtype=float) @ vec
    return float(np.rad2deg(np.arctan2(mapped[1], mapped[0])) % 180.0)


def local_orientation_error_deg(center_x_cm, center_y_cm, *, orientation_deg, true_distance_cm, scenario, eps_cm):
    jac = numerical_jacobian(center_x_cm, center_y_cm, true_distance_cm=true_distance_cm, scenario=scenario, eps_cm=eps_cm)
    theta_out = transformed_orientation_deg(jac, orientation_deg)
    return float(abs(orientation_difference_deg(theta_out, orientation_deg)))


def polar_decompose_error(center_x_cm, center_y_cm, *, true_distance_cm, scenario):
    x_true = np.asarray(center_x_cm, dtype=float)
    y_true = np.asarray(center_y_cm, dtype=float)
    x_map, y_map = mapping_from_true_screen(x_true, y_true, true_distance_cm=true_distance_cm, scenario=scenario)
    dx = x_map - x_true
    dy = y_map - y_true
    radius = np.sqrt(x_true**2 + y_true**2)
    radial_unit_x = np.divide(x_true, radius, out=np.zeros_like(x_true), where=radius > 1e-12)
    radial_unit_y = np.divide(y_true, radius, out=np.zeros_like(y_true), where=radius > 1e-12)
    tangential_unit_x = -radial_unit_y
    tangential_unit_y = radial_unit_x
    radial_error = dx * radial_unit_x + dy * radial_unit_y
    tangential_error = dx * tangential_unit_x + dy * tangential_unit_y
    displacement = np.sqrt(dx**2 + dy**2)
    normalized = np.divide(displacement, radius, out=np.zeros_like(displacement), where=radius > 1e-12)
    return {
        "dx_cm": dx,
        "dy_cm": dy,
        "radius_cm": radius,
        "displacement_cm": displacement,
        "radial_error_cm": radial_error,
        "tangential_error_cm": tangential_error,
        "normalized_error": normalized,
    }


scenario_order = [
    "exact_true_distance",
    "exact_distance_minus_5cm",
    "exact_distance_plus_5cm",
    "planar_no_correction",
]

scenario_labels = {
    "exact_true_distance": "Exact spherical correction, correct distance",
    "exact_distance_minus_5cm": "Exact spherical correction, assumed distance 5 cm too short",
    "exact_distance_plus_5cm": "Exact spherical correction, assumed distance 5 cm too long",
    "planar_no_correction": "No spherical correction on the screen plane",
}

## 5–6. Parameterize Effective FOV Centers and Generate Gaze Shifts

We evaluate two related objects:

- **effective-center surfaces** over the screen from $0^\circ$ to $10^\circ$ in azimuth and elevation,
- **session trajectories** that start from multiple baseline FOV centers and then move under horizontal, vertical, or diagonal gaze drift.

This lets us separate:

- where the geometry is intrinsically more sensitive,
- how much a given session sequence explores that sensitive region.

In [4]:
effective_center_vals_deg = np.linspace(-50.0, 50.0, 41)
center_az_grid_deg, center_el_grid_deg = np.meshgrid(effective_center_vals_deg, effective_center_vals_deg, indexing="xy")
center_grid_df = pd.DataFrame(
    {
        "center_az_deg": center_az_grid_deg.ravel(),
        "center_el_deg": center_el_grid_deg.ravel(),
    }
)

baseline_centers_deg = {
    "centered": (0.0, 0.0),
    "mild_horizontal": (2.5, 0.0),
    "mild_vertical": (0.0, 2.5),
    "mild_diagonal": (2.5, 2.5),
    "strong_horizontal": (5.0, 0.0),
    "strong_diagonal": (5.0, 5.0),
}

baseline_centers_df = pd.DataFrame(
    {
        "baseline_label": list(baseline_centers_deg.keys()),
        "baseline_az_deg": [value[0] for value in baseline_centers_deg.values()],
        "baseline_el_deg": [value[1] for value in baseline_centers_deg.values()],
    }
)

baseline_centers_df

,baseline_label,baseline_az_deg,baseline_el_deg
0,centered,0.0,0.0
1,mild_horizontal,2.5,0.0
2,mild_vertical,0.0,2.5
3,mild_diagonal,2.5,2.5
4,strong_horizontal,5.0,0.0
5,strong_diagonal,5.0,5.0


## 7–10. Project Coordinates, Compute Mapping Errors, and Simulate Session Drift

The next code block does four jobs:

1. converts effective FOV centers from angular coordinates to physical screen coordinates,
2. computes the local orientation distortion using a numerical Jacobian,
3. summarizes spatial displacement over a local image patch around each center,
4. simulates multi-session trajectories from multiple starting FOV centers and aggregates the errors.

Two geometry-only predictions are especially important:

- if the distance is wrong but the inverse remains exactly spherical, local orientation error should stay near zero,
- if the spherical correction is skipped, local orientation error should grow with eccentricity and depend on orientation.

In [5]:
patch_offsets_deg = np.linspace(-0.5 * nominal_patch_extent_deg, 0.5 * nominal_patch_extent_deg, patch_resolution)
patch_az_offset_deg, patch_el_offset_deg = np.meshgrid(patch_offsets_deg, patch_offsets_deg, indexing="xy")


def summarize_local_geometry(center_az_deg, center_el_deg, *, scenario):
    center_x_cm, center_y_cm = angles_deg_to_screen_cm_exact(center_az_deg, center_el_deg, true_distance_cm)
    jac = numerical_jacobian(
        center_x_cm,
        center_y_cm,
        true_distance_cm=true_distance_cm,
        scenario=scenario,
        eps_cm=jacobian_eps_cm,
    )
    orientation_errors = [
        local_orientation_error_deg(
            center_x_cm,
            center_y_cm,
            orientation_deg=ori,
            true_distance_cm=true_distance_cm,
            scenario=scenario,
            eps_cm=jacobian_eps_cm,
        )
        for ori in orientation_grid_deg
    ]
    x_map, y_map = mapping_from_true_screen(center_x_cm, center_y_cm, true_distance_cm=true_distance_cm, scenario=scenario)
    displacement = np.sqrt((x_map - center_x_cm) ** 2 + (y_map - center_y_cm) ** 2)
    singular_values = np.linalg.svd(jac, compute_uv=False)
    return {
        "center_az_deg": float(center_az_deg),
        "center_el_deg": float(center_el_deg),
        "scenario": scenario,
        "center_displacement_cm": float(displacement),
        "max_orientation_error_deg": float(np.max(orientation_errors)),
        "median_orientation_error_deg": float(np.median(orientation_errors)),
        "orientation_error_horizontal_deg": float(orientation_errors[0]),
        "orientation_error_oblique_45_deg": float(orientation_errors[int(45 / 5)]),
        "orientation_error_vertical_deg": float(orientation_errors[int(90 / 5)]),
        "jacobian_det": float(np.linalg.det(jac)),
        "jacobian_scale_major": float(np.max(singular_values)),
        "jacobian_scale_minor": float(np.min(singular_values)),
    }



def summarize_patch_geometry(center_az_deg, center_el_deg, *, scenario):
    patch_az_deg = center_az_deg + patch_az_offset_deg
    patch_el_deg = center_el_deg + patch_el_offset_deg
    true_x_cm, true_y_cm = angles_deg_to_screen_cm_exact(patch_az_deg, patch_el_deg, true_distance_cm)
    map_x_cm, map_y_cm = mapping_from_true_screen(true_x_cm, true_y_cm, true_distance_cm=true_distance_cm, scenario=scenario)
    errors = polar_decompose_error(true_x_cm, true_y_cm, true_distance_cm=true_distance_cm, scenario=scenario)
    return {
        "patch_mean_displacement_cm": float(np.mean(errors["displacement_cm"])),
        "patch_median_displacement_cm": float(np.median(errors["displacement_cm"])),
        "patch_max_displacement_cm": float(np.max(errors["displacement_cm"])),
        "patch_mean_radial_error_cm": float(np.mean(errors["radial_error_cm"])),
        "patch_mean_abs_tangential_error_cm": float(np.mean(np.abs(errors["tangential_error_cm"]))),
        "patch_max_abs_tangential_error_cm": float(np.max(np.abs(errors["tangential_error_cm"]))),
        "patch_mean_normalized_error": float(np.mean(errors["normalized_error"])),
        "patch_p95_displacement_cm": float(np.percentile(errors["displacement_cm"], 95)),
        "patch_corner_displacement_cm": float(errors["displacement_cm"][0, 0]),
        "patch_center_displacement_cm": float(errors["displacement_cm"][patch_resolution // 2, patch_resolution // 2]),
    }


surface_rows = []
for scenario in scenario_order:
    for row in center_grid_df.itertuples(index=False):
        surface_rows.append(summarize_local_geometry(row.center_az_deg, row.center_el_deg, scenario=scenario))

surface_df = pd.DataFrame(surface_rows)
surface_df.to_csv(TABLE_DIR / f"{NOTEBOOK_NAME}_effective_center_surfaces.csv", index=False)

session_rows = []
for baseline_label, (baseline_az_deg, baseline_el_deg) in baseline_centers_deg.items():
    for path_label, path_values in gaze_paths.items():
        for session_index, (shift_az_deg, shift_el_deg) in enumerate(path_values):
            effective_az_deg = baseline_az_deg + float(shift_az_deg)
            effective_el_deg = baseline_el_deg + float(shift_el_deg)
            for scenario in scenario_order:
                row = {
                    "baseline_label": baseline_label,
                    "baseline_az_deg": baseline_az_deg,
                    "baseline_el_deg": baseline_el_deg,
                    "path_label": path_label,
                    "session_index": session_index,
                    "gaze_shift_az_deg": float(shift_az_deg),
                    "gaze_shift_el_deg": float(shift_el_deg),
                    "gaze_shift_magnitude_deg": float(np.sqrt(shift_az_deg**2 + shift_el_deg**2)),
                    "effective_center_az_deg": effective_az_deg,
                    "effective_center_el_deg": effective_el_deg,
                    "effective_center_radius_deg": float(np.sqrt(effective_az_deg**2 + effective_el_deg**2)),
                    "scenario": scenario,
                }
                row.update(summarize_local_geometry(effective_az_deg, effective_el_deg, scenario=scenario))
                row.update(summarize_patch_geometry(effective_az_deg, effective_el_deg, scenario=scenario))
                session_rows.append(row)

session_df = pd.DataFrame(session_rows)
session_df.to_csv(TABLE_DIR / f"{NOTEBOOK_NAME}_session_metrics.csv", index=False)

surface_df.head(), session_df.head()

(   center_az_deg  center_el_deg             scenario  center_displacement_cm  \
 0          -50.0          -50.0  exact_true_distance            7.944109e-15   
 1          -47.5          -50.0  exact_true_distance            3.552714e-15   
 2          -45.0          -50.0  exact_true_distance            3.552714e-15   
 3          -42.5          -50.0  exact_true_distance            7.105427e-15   
 4          -40.0          -50.0  exact_true_distance            0.000000e+00   
 
    max_orientation_error_deg  median_orientation_error_deg  \
 0               1.074341e-11                  5.087486e-12   
 1               1.838885e-11                  7.638334e-12   
 2               1.756462e-11                  7.638334e-12   
 3               5.385914e-12                  2.550848e-12   
 4               1.838885e-11                  7.631229e-12   
 
    orientation_error_horizontal_deg  orientation_error_oblique_45_deg  \
 0                      1.017497e-11                      

## 11–12. Visualize Error Surfaces and Compare FOV-Center Sensitivity

We plot two kinds of geometry-only summaries:

- **effective-center surfaces**: what happens if the current FOV center sits at a given location on the screen,
- **session trajectories**: how geometry changes as gaze shifts carry different starting centers across sessions.

The central diagnostic is the maximum local orientation error over the orientation grid at each effective center.

In [6]:
surface_plot_metric = "max_orientation_error_deg"

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, scenario in zip(axes.ravel(), scenario_order):
    subset = surface_df[surface_df["scenario"] == scenario]
    pivot = subset.pivot(index="center_el_deg", columns="center_az_deg", values=surface_plot_metric)
    sns.heatmap(pivot, ax=ax, cmap="magma", cbar=True)
    ax.set_title(scenario_labels[scenario])
    ax.set_xlabel("effective center azimuth (deg)")
    ax.set_ylabel("effective center elevation (deg)")
fig.tight_layout()
fig.savefig(FIG_DIR / f"{NOTEBOOK_NAME}_orientation_error_surfaces.png", dpi=200, bbox_inches="tight")
plt.close(fig)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
reference = surface_df[surface_df["scenario"] == "exact_true_distance"].pivot(
    index="center_el_deg",
    columns="center_az_deg",
    values=surface_plot_metric,
)
for ax, scenario in zip(axes.ravel(), scenario_order[1:]):
    subset = surface_df[surface_df["scenario"] == scenario].pivot(
        index="center_el_deg",
        columns="center_az_deg",
        values=surface_plot_metric,
    )
    sns.heatmap(subset - reference, ax=ax, cmap="coolwarm", center=0.0, cbar=True)
    ax.set_title(f"{scenario_labels[scenario]} minus exact")
    ax.set_xlabel("effective center azimuth (deg)")
    ax.set_ylabel("effective center elevation (deg)")
fig.tight_layout()
fig.savefig(FIG_DIR / f"{NOTEBOOK_NAME}_orientation_error_difference_from_exact.png", dpi=200, bbox_inches="tight")
plt.close(fig)

trajectory_subset = session_df[session_df["path_label"] == "diagonal"].copy()
fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
sns.lineplot(
    data=trajectory_subset,
    x="gaze_shift_magnitude_deg",
    y="max_orientation_error_deg",
    hue="baseline_label",
    style="scenario",
    ax=axes[0],
)
axes[0].set_title("Session-wise geometric orientation error along diagonal gaze drift")
axes[0].set_ylabel("max local orientation error (deg)")
axes[0].legend(frameon=False, fontsize=8, ncol=2)

sns.lineplot(
    data=trajectory_subset,
    x="gaze_shift_magnitude_deg",
    y="patch_mean_displacement_cm",
    hue="baseline_label",
    style="scenario",
    ax=axes[1],
)
axes[1].set_title("Session-wise mean patch displacement along diagonal gaze drift")
axes[1].set_xlabel("gaze-shift magnitude (deg)")
axes[1].set_ylabel("mean displacement over local patch (cm)")
axes[1].legend(frameon=False, fontsize=8, ncol=2)
fig.tight_layout()
fig.savefig(FIG_DIR / f"{NOTEBOOK_NAME}_session_diagonal_drift_curves.png", dpi=200, bbox_inches="tight")
plt.close(fig)

center_sensitivity = (
    session_df.groupby(["scenario", "baseline_label", "path_label"], as_index=False)
    .agg(
        max_orientation_error_deg=("max_orientation_error_deg", "max"),
        median_orientation_error_deg=("median_orientation_error_deg", "median"),
        max_patch_displacement_cm=("patch_max_displacement_cm", "max"),
        mean_patch_displacement_cm=("patch_mean_displacement_cm", "mean"),
    )
)
center_sensitivity.to_csv(TABLE_DIR / f"{NOTEBOOK_NAME}_center_sensitivity_summary.csv", index=False)
center_sensitivity.head(12)

,scenario,baseline_label,path_label,max_orientation_error_deg,median_orientation_error_deg,max_patch_displacement_cm,mean_patch_displacement_cm
0,exact_distance_minus_5cm,centered,diagonal,9.947598e-13,9.237056e-14,1.396761,0.524246
1,exact_distance_minus_5cm,centered,horizontal,3.126388e-13,4.263256e-14,1.297673,0.526274
2,exact_distance_minus_5cm,centered,vertical,3.126388e-13,4.263256e-14,1.297673,0.526274
3,exact_distance_minus_5cm,mild_diagonal,diagonal,1.477929e-12,2.273737e-13,1.731579,0.787260
4,exact_distance_minus_5cm,mild_diagonal,horizontal,3.979039e-13,1.136868e-13,1.595797,0.741143
5,exact_distance_minus_5cm,mild_diagonal,vertical,9.663381e-13,2.344791e-13,1.595797,0.741143
6,exact_distance_minus_5cm,mild_horizontal,diagonal,1.023182e-12,1.421085e-13,1.571515,0.667257
7,exact_distance_minus_5cm,mild_horizontal,horizontal,3.126388e-13,4.263256e-14,1.525297,0.705683
8,exact_distance_minus_5cm,mild_horizontal,vertical,9.663381e-13,1.136868e-13,1.377976,0.569724
9,exact_distance_minus_5cm,mild_vertical,diagonal,7.673862e-13,1.847411e-13,1.571515,0.667257


## 13. Fit Simple Error-Growth Models

We fit compact regression models to quantify whether the geometry-only error grows mainly with:

- gaze-shift magnitude,
- session index,
- effective-center eccentricity,
- interactions between shift magnitude and eccentricity.

The goal is interpretability, not a fully optimized predictive model.

In [7]:
def fit_linear_model(frame, target):
    design = pd.DataFrame(
        {
            "intercept": 1.0,
            "gaze_shift_magnitude_deg": frame["gaze_shift_magnitude_deg"],
            "gaze_shift_sq": frame["gaze_shift_magnitude_deg"] ** 2,
            "session_index": frame["session_index"],
            "effective_center_radius_deg": frame["effective_center_radius_deg"],
            "interaction_shift_radius": frame["gaze_shift_magnitude_deg"] * frame["effective_center_radius_deg"],
        }
    )
    x = design.to_numpy(float)
    y = frame[target].to_numpy(float)
    beta, *_ = np.linalg.lstsq(x, y, rcond=None)
    fitted = x @ beta
    residual = y - fitted
    ss_res = float(np.sum(residual**2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 1e-12 else np.nan
    coef = pd.DataFrame(
        {
            "term": design.columns,
            "coefficient": beta,
        }
    )
    return coef, r2


model_rows = []
for scenario in scenario_order:
    subset = session_df[session_df["scenario"] == scenario].copy()
    for target in ["max_orientation_error_deg", "patch_mean_displacement_cm"]:
        coef, r2 = fit_linear_model(subset, target)
        coef["scenario"] = scenario
        coef["target"] = target
        coef["r_squared"] = r2
        model_rows.append(coef)

model_df = pd.concat(model_rows, ignore_index=True)
model_df.to_csv(TABLE_DIR / f"{NOTEBOOK_NAME}_regression_coefficients.csv", index=False)
model_df.head(20)

,term,coefficient,scenario,target,r_squared
0,intercept,6.601204e-14,exact_true_distance,max_orientation_error_deg,NaN
1,gaze_shift_magnitude_deg,-9.296048e-15,exact_true_distance,max_orientation_error_deg,NaN
2,gaze_shift_sq,1.535598e-15,exact_true_distance,max_orientation_error_deg,NaN
3,session_index,-1.859210e-14,exact_true_distance,max_orientation_error_deg,NaN
4,effective_center_radius_deg,4.361395e-14,exact_true_distance,max_orientation_error_deg,NaN
5,interaction_shift_radius,2.276281e-15,exact_true_distance,max_orientation_error_deg,NaN
6,intercept,4.058509e-17,exact_true_distance,patch_mean_displacement_cm,NaN
7,gaze_shift_magnitude_deg,-1.086903e-18,exact_true_distance,patch_mean_displacement_cm,NaN
8,gaze_shift_sq,-2.263799e-19,exact_true_distance,patch_mean_displacement_cm,NaN
9,session_index,-2.173807e-18,exact_true_distance,patch_mean_displacement_cm,NaN


## 14. Export Summary Tables and Reproducible Artifacts

The final cell writes a compact artifact manifest and prints the main geometry-only conclusion.

The expected qualitative result is:

- wrong distance inside the exact spherical inverse produces large spatial scale error but essentially no local orientation rotation,
- no spherical correction produces small but nonzero local orientation distortion that grows with eccentricity and therefore with effective FOV center and session gaze drift.

In [8]:
artifact_manifest = pd.DataFrame(
    {
        "artifact": [
            f"{NOTEBOOK_NAME}_effective_center_surfaces.csv",
            f"{NOTEBOOK_NAME}_session_metrics.csv",
            f"{NOTEBOOK_NAME}_center_sensitivity_summary.csv",
            f"{NOTEBOOK_NAME}_regression_coefficients.csv",
            f"{NOTEBOOK_NAME}_orientation_error_surfaces.png",
            f"{NOTEBOOK_NAME}_orientation_error_difference_from_exact.png",
            f"{NOTEBOOK_NAME}_session_diagonal_drift_curves.png",
        ],
        "location": [
            str(TABLE_DIR / f"{NOTEBOOK_NAME}_effective_center_surfaces.csv"),
            str(TABLE_DIR / f"{NOTEBOOK_NAME}_session_metrics.csv"),
            str(TABLE_DIR / f"{NOTEBOOK_NAME}_center_sensitivity_summary.csv"),
            str(TABLE_DIR / f"{NOTEBOOK_NAME}_regression_coefficients.csv"),
            str(FIG_DIR / f"{NOTEBOOK_NAME}_orientation_error_surfaces.png"),
            str(FIG_DIR / f"{NOTEBOOK_NAME}_orientation_error_difference_from_exact.png"),
            str(FIG_DIR / f"{NOTEBOOK_NAME}_session_diagonal_drift_curves.png"),
        ],
    }
)
artifact_manifest.to_csv(TABLE_DIR / f"{NOTEBOOK_NAME}_artifact_manifest.csv", index=False)

headline = (
    surface_df.groupby("scenario", as_index=False)
    .agg(
        max_orientation_error_deg=("max_orientation_error_deg", "max"),
        median_orientation_error_deg=("median_orientation_error_deg", "median"),
        max_center_displacement_cm=("center_displacement_cm", "max"),
    )
    .sort_values("max_orientation_error_deg", ascending=False)
)

print("Geometry-only headline summary")
print(headline.to_string(index=False))
print()
print("Key interpretation:")
print("- Exact spherical correction with the wrong distance changes scale but should not rotate local orientation.")
print("- Planar no-correction introduces the geometric orientation distortion that grows away from the screen center.")

artifact_manifest

Geometry-only headline summary
                scenario  max_orientation_error_deg  median_orientation_error_deg  max_center_displacement_cm
    planar_no_correction               4.278629e+01                  6.621634e+00                1.547920e+01
     exact_true_distance               2.569323e-11                  1.278977e-12                7.944109e-15
exact_distance_minus_5cm               2.290790e-11                  1.428191e-12                1.102014e+01
 exact_distance_plus_5cm               2.290790e-11                  1.428191e-12                1.102014e+01

Key interpretation:
- Exact spherical correction with the wrong distance changes scale but should not rotate local orientation.
- Planar no-correction introduces the geometric orientation distortion that grows away from the screen center.


,artifact,location
0,geometry_only_wrong_spherical_mapping_effectiv...,/Users/tobiasr/Documents/GitHub/model-gabordri...
1,geometry_only_wrong_spherical_mapping_session_...,/Users/tobiasr/Documents/GitHub/model-gabordri...
2,geometry_only_wrong_spherical_mapping_center_s...,/Users/tobiasr/Documents/GitHub/model-gabordri...
3,geometry_only_wrong_spherical_mapping_regressi...,/Users/tobiasr/Documents/GitHub/model-gabordri...
4,geometry_only_wrong_spherical_mapping_orientat...,/Users/tobiasr/Documents/GitHub/model-gabordri...
5,geometry_only_wrong_spherical_mapping_orientat...,/Users/tobiasr/Documents/GitHub/model-gabordri...
6,geometry_only_wrong_spherical_mapping_session_...,/Users/tobiasr/Documents/GitHub/model-gabordri...
